In [1]:
from iconclass import init
import math

ic = init()  # root Notation


def walk(node):
    """Yield node and all its descendants recursively."""
    yield node
    for child in node:
        yield from walk(child)


# Build corpus: a list of (node, tokens)
corpus = []
for node in walk(ic):
    try:
        label = node("en").lower()
    except:
        label = ""
    tokens = label.split()
    corpus.append((node, tokens))

N = len(corpus)  # total number of documents

In [2]:
from collections import defaultdict

df = defaultdict(int)

for _, tokens in corpus:
    unique_tokens = set(tokens)
    for t in unique_tokens:
        df[t] += 1

In [3]:
k1 = 1.5
b = 0.75

# Average document length
avgdl = sum(len(tokens) for _, tokens in corpus) / N


def bm25_score(query_tokens, doc_tokens):
    """Compute BM25 score for a single label/doc."""
    score = 0.0
    doc_len = len(doc_tokens)

    for q in query_tokens:
        f = doc_tokens.count(q)  # term frequency in this document
        if f == 0:
            continue
        
        # Document frequency
        df_q = df.get(q, 0)
        if df_q == 0:
            continue
        
        # IDF (classic BM25 IDF)
        idf = math.log( (N - df_q + 0.5) / (df_q + 0.5) + 1 )

        # Term weight
        denom = f + k1 * (1 - b + b * (doc_len / avgdl))
        score += idf * ( (f * (k1 + 1)) / denom )

    return score

In [4]:
def search_iconclass_bm25(query):
    query_tokens = query.lower().split()
    results = []

    for node, tokens in corpus:
        score = bm25_score(query_tokens, tokens)
        if score > 0:
            results.append((score, node))

    # Sort by score descending
    results.sort(key=lambda x: x[0], reverse=True)
    return results

In [5]:
results = search_iconclass_bm25("sleeping lion")

for score, node in results[:20]:  # show first 20
    notation = repr(node).split()[0]
    print(f"{score:.4f}  {notation} → {node('en')}")

14.8165  25F23(LION)(+46) → beasts of prey, predatory animals: lion (+ sleeping animal(s))
12.2803  25FF23(LION)(+746) → beasts of prey, predatory animals: lion - FF - fabulous animals (+ sleeping animal(s))
12.2803  25FF23(LION)(+746) → beasts of prey, predatory animals: lion - FF - fabulous animals (+ sleeping animal(s))
12.2803  25FF23(LION)(+746) → beasts of prey, predatory animals: lion - FF - fabulous animals (+ sleeping animal(s))
12.2803  25FF23(LION)(+746) → beasts of prey, predatory animals: lion - FF - fabulous animals (+ sleeping animal(s))
12.2803  25FF23(LION)(+746) → beasts of prey, predatory animals: lion - FF - fabulous animals (+ sleeping animal(s))
12.2803  25FF23(LION)(+746) → beasts of prey, predatory animals: lion - FF - fabulous animals (+ sleeping animal(s))
10.5926  43C111232 → lion hunt
10.5926  92I74911 → Nemean lion
9.6874  31B11 → sleeping in bed
9.6874  31B12 → sleeping in chair
9.4898  43C1112321 → hunter fighting a lion
9.4898  43C111232(+0) → lion hunt 

In [6]:
if __name__ == "__main__":
    query = input("Enter your Iconclass search query: ")

    results = search_iconclass_bm25(query)

    print("\nTop results:\n")

    for score, node in results[:20]:  # show first 20 results
        notation = repr(node).split()[0]
        try:
            label = node("en")
        except:
            label = "(no label)"
        print(f"{score:.4f}  {notation} → {label}")


Top results:

10.5926  43C111232 → lion hunt
10.5926  92I74911 → Nemean lion
9.4898  43C1112321 → hunter fighting a lion
9.4898  43C111232(+0) → lion hunt (+ variant)
9.4898  43C111232(+4111) → lion hunt (+ spear)
9.4898  43C111232(+4113) → lion hunt (+ blow-pipe)
9.4898  43C111232(+4114) → lion hunt (+ boomerang)
9.4898  43C111232(+412) → lion hunt (+ firearms)
9.4898  43C111232(+413) → lion hunt (+ lasso)
9.4898  43C111232(+414) → lion hunt (+ trap)
9.4898  43C111232(+4141) → lion hunt (+ snare)
9.4898  43C111232(+4142) → lion hunt (+ pitfall)
9.4898  43C111232(+415) → lion hunt (+ net)
9.4898  43C111232(+4161) → lion hunt (+ fishing-rod)
9.4898  43C111232(+41611) → lion hunt (+ fishing-tackle)
9.4898  43C111232(+41612) → lion hunt (+ fishing-line)
9.4898  43C111232(+41613) → lion hunt (+ float)
9.4898  43C111232(+41614) → lion hunt (+ fish-hook)
9.4898  43C111232(+4162) → lion hunt (+ harpoon)
9.4898  43C111232(+41621) → lion hunt (+ trident)
